# Phase II · DQN Prompt Selector — Training & Analysis

**Project:** Adaptive Customer Service Chatbot  
**Phase:** II — Full Reinforcement Learning (MDP)  
**Algorithm:** Deep Q-Network (DQN)  

---

## Pipeline Overview

```
Conversation State (15-dim)
    │
    ▼
┌─────────────────────┐
│   DQN Agent         │   ← learns Q(s, a) for all 5 strategies
│   argmax Q(s, ·)    │
└────────┬────────────┘
         │  action index (0–4)
         ▼
┌─────────────────────┐
│  Strategy Prompt    │   ← rich system instruction (Ask / Solve / Empathize /
│  Template Library   │     Escalate / Close)  +  live conversation context
└────────┬────────────┘
         │  formatted prompt
         ▼
┌─────────────────────┐
│  LLM  (Mock / Real) │   ← generates natural-language response
└────────┬────────────┘
         │  response text
         ▼
┌─────────────────────┐
│  Environment Sim    │   ← updates sentiment, frustration, info_gathered
└────────┬────────────┘
         │  new state + reward
         ▼
    Replay Buffer  →  Gradient Update
```

---
### What makes Phase II different from Phase I (Bandits)

| Aspect | Phase I (Bandits) | Phase II (DQN) |
|--------|-------------------|----------------|
| Strategy selection | One step at a time | Plans over full conversation |
| State | 4-dim snapshot | 15-dim with action history |
| Memory | None | 10 000-step replay buffer |
| Credit assignment | Immediate reward only | Bellman: R + γ·max Q(s') |
| Exploration | TS/UCB | ε-greedy with decay |

In [5]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from mdp_phase2.environment      import MDPCustomerServiceEnv
from mdp_phase2.agents.dqn_agent import DQNAgent
from mdp_phase2.reward           import RewardShaper
from mdp_phase2.mock_llm         import MockLLM
from mdp_phase2.train            import train_dqn, evaluate, run_demo_conversation, TrainConfig
from mdp_phase2.strategy_prompts import describe_strategies, STRATEGY_NAMES, STRATEGY_LABELS, STRATEGY_COLORS

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = list(STRATEGY_COLORS.values())

print('All imports OK ✓')
print(f'State dimension  : {MDPCustomerServiceEnv.STATE_DIM}')
print(f'Number of actions: {MDPCustomerServiceEnv.NUM_ACTIONS}')
print(f'Strategies       : {STRATEGY_LABELS}')

All imports OK ✓
State dimension  : 15
Number of actions: 5
Strategies       : ['Ask', 'Solve', 'Empathize', 'Escalate', 'Close']


## 1 · Strategy Prompt Library

These are the 5 system prompts the DQN agent selects among.  
Each prompt shapes the LLM's tone, goal, and behaviour for that turn.

In [6]:
describe_strategies()

  STRATEGY PROMPT LIBRARY — Phase II RL Prompt Selector

[0] Ask           (ask_info)
    When to use: Gather missing details to diagnose the issue accurately.
    Prompt len : 519 chars

[1] Solve         (provide_solution)
    When to use: Deliver a direct, actionable fix to the customer's problem.
    Prompt len : 475 chars

[2] Empathize     (empathize)
    When to use: Acknowledge the customer's frustration and rebuild rapport.
    Prompt len : 600 chars

[3] Escalate      (escalate)
    When to use: Hand off to a senior agent or specialist when the issue exceeds scope.
    Prompt len : 605 chars

[4] Close         (close)
    When to use: Wrap up the conversation after successful resolution.
    Prompt len : 454 chars


## 2 · State Space Visualisation

The 15-dimensional state vector gives the DQN agent **temporal context** that  
contextual bandits completely lack.

In [7]:
env = MDPCustomerServiceEnv('twitter')
state = env.reset()

labels = [
    'Sentiment', 'Frustration', 'Info Gathered', 'Turn (norm)',
    'Last: Ask', 'Last: Solve', 'Last: Empathize', 'Last: Escalate', 'Last: Close',
    'Usage: Ask', 'Usage: Solve', 'Usage: Empathize', 'Usage: Escalate', 'Usage: Close',
    'Repetition Signal'
]

fig, ax = plt.subplots(figsize=(14, 3))
bar_colors = (['#2196F3']*4 + ['#FF9800']*5 + ['#4CAF50']*5 + ['#F44336'])
ax.bar(range(15), state, color=bar_colors, edgecolor='white', linewidth=0.8)
ax.set_xticks(range(15))
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Value')
ax.set_title('Phase II State Vector (15 dims) — Start of Episode: Twitter dataset', fontweight='bold')
ax.set_ylim(0, 1.05)

legend_patches = [
    mpatches.Patch(color='#2196F3', label='Core signals (4)'),
    mpatches.Patch(color='#FF9800', label='Last action one-hot (5)'),
    mpatches.Patch(color='#4CAF50', label='Strategy usage freq (5)'),
    mpatches.Patch(color='#F44336', label='Repetition signal (1)'),
]
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('state_space_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nInitial state (turn 0): {state}')

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 3 · DQN Agent Architecture

In [ ]:
agent = DQNAgent(
    state_dim          = MDPCustomerServiceEnv.STATE_DIM,   # 15
    num_actions        = MDPCustomerServiceEnv.NUM_ACTIONS, # 5
    gamma              = 0.90,    # discount factor
    lr                 = 1e-3,
    eps_start          = 1.0,     # start fully exploring
    eps_end            = 0.05,    # never go below 5% random
    eps_decay          = 0.0015,  # ~700 episodes to reach eps_end
    batch_size         = 64,
    buffer_size        = 10_000,
    target_update_freq = 50,
    hidden             = (128, 64),
)

try:
    agent.online.summary()
except:
    print('DQN Agent (numpy backend)')
    print(f'  Architecture : 15 → 128 → 64 → 5')
    print(f'  Parameters   : {15*128 + 128 + 128*64 + 64 + 64*5 + 5:,}')

print(f'\nReplay buffer capacity: {agent.replay.buf.maxlen:,}')
print(f'Batch size            : {agent.batch_size}')
print(f'Target update every   : {agent.target_update_freq} gradient steps')

## 4 · Training — Twitter Dataset

**Why Twitter first?**  Twitter has the hardest profile: short episodes (max 10 turns),  
frustrated customers (frustration=0.65), low initial sentiment (0.30).  
The agent must learn to balance empathy with efficiency.

In [ ]:
cfg = TrainConfig(
    n_episodes   = 3000,
    eval_every   = 100,
    eval_episodes= 50,
    print_every  = 300,
    smooth_window= 50,
    seed         = 42,
    save_path    = 'saved_models',
)

result_twitter = train_dqn(agent, dataset='twitter', cfg=cfg)

## 5 · Learning Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# ── Episode reward ─────────────────────────────────────────────────────────
ax = axes[0, 0]
raw = result_twitter.episode_rewards
smooth = result_twitter.smooth_rewards(50)
ax.plot(raw, alpha=0.2, color='#4C72B0', linewidth=0.5)
offset = len(raw) - len(smooth)
ax.plot(range(offset, offset + len(smooth)), smooth, color='#4C72B0', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('Episode Reward (Twitter)', fontweight='bold')

# ── Success rate over training ─────────────────────────────────────────────
ax = axes[0, 1]
eval_eps   = [ev.episode  for ev in result_twitter.eval_snapshots]
eval_sr    = [ev.success_rate * 100 for ev in result_twitter.eval_snapshots]
eval_esc   = [ev.escalation_rate * 100 for ev in result_twitter.eval_snapshots]
eval_aband = [ev.abandon_rate * 100   for ev in result_twitter.eval_snapshots]
ax.plot(eval_eps, eval_sr,    color='#2CA02C', linewidth=2, marker='o', ms=4, label='Resolved')
ax.plot(eval_eps, eval_esc,   color='#D62728', linewidth=2, marker='s', ms=4, label='Escalated')
ax.plot(eval_eps, eval_aband, color='#FF7F0E', linewidth=2, marker='^', ms=4, label='Abandoned')
ax.set_xlabel('Episode')
ax.set_ylabel('Rate (%)')
ax.set_title('Outcome Rates During Training (Twitter)', fontweight='bold')
ax.legend()
ax.set_ylim(0, 100)

# ── Loss curve ─────────────────────────────────────────────────────────────
ax = axes[1, 0]
losses = result_twitter.losses
# smooth the loss
loss_smooth = np.convolve(losses, np.ones(50)/50, mode='valid')
ax.plot(losses, alpha=0.2, color='#9467BD', linewidth=0.5)
ax.plot(range(49, 49 + len(loss_smooth)), loss_smooth, color='#9467BD', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Huber Loss')
ax.set_title('DQN Training Loss', fontweight='bold')

# ── Strategy usage over training ───────────────────────────────────────────
ax = axes[1, 1]
final_eval = result_twitter.eval_snapshots[-1]
strategy_vals = [final_eval.strategy_dist.get(n, 0) * 100 for n in STRATEGY_NAMES]
bars = ax.bar(STRATEGY_LABELS, strategy_vals, color=COLORS, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, strategy_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Usage (%)')
ax.set_title('Final Strategy Distribution (Twitter)', fontweight='bold')
ax.set_ylim(0, max(strategy_vals) + 8)

plt.suptitle('DQN Training Results — Twitter Dataset', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('dqn_training_twitter.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal evaluation ({cfg.eval_episodes} episodes):")
ev = result_twitter.eval_snapshots[-1]
print(f"  Resolution rate : {ev.success_rate*100:.1f}%")
print(f"  Escalation rate : {ev.escalation_rate*100:.1f}%")
print(f"  Abandon rate    : {ev.abandon_rate*100:.1f}%")
print(f"  Avg reward      : {ev.avg_reward:+.3f}")
print(f"  Avg turns       : {ev.avg_turns:.1f}")

## 6 · Training — Reddit & OpenAssistant Datasets

In [ ]:
# ── Reddit ─────────────────────────────────────────────────────────────────
agent_reddit = DQNAgent(
    state_dim=MDPCustomerServiceEnv.STATE_DIM,
    num_actions=MDPCustomerServiceEnv.NUM_ACTIONS,
    gamma=0.90, lr=1e-3, eps_start=1.0, eps_end=0.05,
    eps_decay=0.0015, batch_size=64, buffer_size=10_000,
    target_update_freq=50, hidden=(128, 64),
)

result_reddit = train_dqn(agent_reddit, dataset='reddit', cfg=cfg)

# ── OpenAssistant ──────────────────────────────────────────────────────────
agent_oa = DQNAgent(
    state_dim=MDPCustomerServiceEnv.STATE_DIM,
    num_actions=MDPCustomerServiceEnv.NUM_ACTIONS,
    gamma=0.90, lr=1e-3, eps_start=1.0, eps_end=0.05,
    eps_decay=0.0015, batch_size=64, buffer_size=10_000,
    target_update_freq=50, hidden=(128, 64),
)

result_oa = train_dqn(agent_oa, dataset='openassistant', cfg=cfg)

## 7 · Cross-Dataset Comparison

In [ ]:
results = {'Twitter': result_twitter, 'Reddit': result_reddit, 'OpenAssistant': result_oa}
agents  = {'Twitter': agent,          'Reddit': agent_reddit,  'OpenAssistant': agent_oa}
dataset_keys = {'Twitter': 'twitter', 'Reddit': 'reddit', 'OpenAssistant': 'openassistant'}
ds_colors = ['#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Success rates ──────────────────────────────────────────────────────────
ax = axes[0]
for i, (name, res) in enumerate(results.items()):
    sr = [ev.success_rate * 100 for ev in res.eval_snapshots]
    ep = [ev.episode            for ev in res.eval_snapshots]
    ax.plot(ep, sr, color=ds_colors[i], linewidth=2, label=name, marker='o', ms=3)
ax.set_xlabel('Episode')
ax.set_ylabel('Resolution Rate (%)')
ax.set_title('Learning Curves — DQN', fontweight='bold')
ax.legend()
ax.set_ylim(0, 100)

# ── Final metrics bar chart ────────────────────────────────────────────────
ax = axes[1]
x  = np.arange(3)
w  = 0.25
final_sr  = [r.eval_snapshots[-1].success_rate*100   for r in results.values()]
final_esc = [r.eval_snapshots[-1].escalation_rate*100 for r in results.values()]
final_ab  = [r.eval_snapshots[-1].abandon_rate*100    for r in results.values()]
ax.bar(x - w, final_sr,  w, color='#2CA02C', label='Resolved')
ax.bar(x,     final_esc, w, color='#D62728', label='Escalated')
ax.bar(x + w, final_ab,  w, color='#FF7F0E', label='Abandoned')
ax.set_xticks(x)
ax.set_xticklabels(list(results.keys()))
ax.set_ylabel('Rate (%)')
ax.set_title('Final Outcome Rates — DQN', fontweight='bold')
ax.legend()

# ── Average turns ──────────────────────────────────────────────────────────
ax = axes[2]
final_turns = [r.eval_snapshots[-1].avg_turns for r in results.values()]
bars = ax.bar(list(results.keys()), final_turns, color=ds_colors, edgecolor='white')
for bar, val in zip(bars, final_turns):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}', ha='center', fontsize=11)
ax.set_ylabel('Avg Turns per Episode')
ax.set_title('Efficiency — DQN', fontweight='bold')

plt.suptitle('DQN Cross-Dataset Results', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('dqn_cross_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Q-Value Analysis

Visualise what the trained DQN agent has learned:  
which strategies it prefers in different conversation states.

In [ ]:
# create a few representative conversation states to probe Q-values
probe_states = {
    'Frustrated customer\n(high frustration, low sentiment, low info)': 
        np.array([0.20, 0.85, 0.10, 0.10] + [0,0,0,0,0] + [0,0,0,0,0] + [0], dtype=np.float32),
    'Information gathering phase\n(mid sentiment, mid frustration, partial info)':
        np.array([0.55, 0.40, 0.40, 0.30] + [1,0,0,0,0] + [0.3,0,0,0,0] + [0], dtype=np.float32),
    'Ready to solve\n(good info gathered, decent sentiment)': 
        np.array([0.70, 0.25, 0.80, 0.50] + [0,1,0,0,0] + [0.2,0.2,0,0,0] + [0], dtype=np.float32),
    'Near resolution\n(very high info, positive customer)': 
        np.array([0.80, 0.15, 0.95, 0.70] + [0,1,0,0,0] + [0.1,0.4,0.1,0,0] + [0], dtype=np.float32),
    'Escalation candidate\n(very frustrated, stuck after many turns)':
        np.array([0.20, 0.95, 0.30, 0.85] + [0,1,0,0,0] + [0.1,0.3,0.1,0,0] + [1], dtype=np.float32),
}

fig, axes = plt.subplots(1, len(probe_states), figsize=(16, 4))

for ax, (title, state) in zip(axes, probe_states.items()):
    q_vals = agent.get_q_values(state)
    bar_colors = [COLORS[i] for i in range(len(STRATEGY_LABELS))]
    bars = ax.bar(STRATEGY_LABELS, q_vals, color=bar_colors, edgecolor='white')
    # highlight the greedy action
    best = int(np.argmax(q_vals))
    bars[best].set_edgecolor('black')
    bars[best].set_linewidth(2.5)
    ax.set_title(title, fontsize=8, fontweight='bold')
    ax.set_ylabel('Q-Value')
    ax.set_xticklabels(STRATEGY_LABELS, rotation=30, ha='right', fontsize=8)

plt.suptitle('Q-Values for Representative Conversation States — DQN (Twitter)',
             fontsize=12, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('dqn_q_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 9 · Demo Conversation

Full pipeline walkthrough: **State → DQN selects strategy → Prompt formatted → LLM generates response**

In [ ]:
demo = run_demo_conversation(agent, dataset='twitter', verbose=True)

## 10 · Summary

In [ ]:
print('=' * 60)
print('  DQN PHASE II — TRAINING SUMMARY')
print('=' * 60)
for ds_name, res in results.items():
    ev = res.eval_snapshots[-1]
    print(f'\n  {ds_name}')
    print(f'    Resolution rate : {ev.success_rate*100:.1f}%')
    print(f'    Escalation rate : {ev.escalation_rate*100:.1f}%')
    print(f'    Avg reward      : {ev.avg_reward:+.3f}')
    print(f'    Avg turns       : {ev.avg_turns:.1f}')
    print(f'    Training time   : {res.train_time_s:.1f}s')
    usage = res.aggregate_strategy_usage()
    top_strat = max(usage, key=usage.get)
    print(f'    Most used strat : {top_strat} ({usage[top_strat]*100:.1f}%)')
print('\n  See comparison_phase2.ipynb for DQN vs A2C vs PPO analysis')
print('=' * 60)